In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
import torch
from transformers import AutoModel
import librosa

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

# Build audio lookup
audio_map = {}
for f in os.listdir(AUDIO_DIR):
    if not f.endswith(('.m4a', '.wav', '.mp3')):
        continue
    base = f.rsplit('.', 1)[0]
    if ',' in base:
        base = base.split(',')[0]
    audio_map[base] = os.path.join(AUDIO_DIR, f)

# Build label lookup
label_map = {}
for f in os.listdir(LABEL_DIR):
    if f.endswith('.csv'):
        label_map[f.replace('.csv', '')] = os.path.join(LABEL_DIR, f)

overlap = sorted(set(audio_map.keys()) & set(label_map.keys()) - done)
print(f'Audio: {len(audio_map)} | Labels: {len(label_map)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')


In [ ]:
# Load WavLM on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
SR = 16000
MAX_SEG = int(3.0 * SR)  # 3 seconds max per word
print(f'WavLM ready! MAX_SEG={MAX_SEG} samples ({MAX_SEG/SR:.1f}s)')


In [ ]:
# OOM-safe per-word extraction
def parse_timestamp(ts):
    ts = str(ts).strip().strip('[]')
    p = ts.split(',')
    return float(p[0]), float(p[1])

def extract_video_words(audio_path, word_times, max_batch=32):
    n = len(word_times)
    if n == 0:
        return None

    # Load audio ONCE
    y_full, _ = librosa.load(audio_path, sr=SR, mono=True)
    n_samples = len(y_full)

    all_emb = []
    for batch_start in range(0, n, max_batch):
        batch_times = word_times[batch_start:batch_start+max_batch]
        
        # Build batch with FIXED length padding (max MAX_SEG)
        batch_padded = np.zeros((len(batch_times), MAX_SEG), dtype=np.float32)
        valid_mask = np.zeros(len(batch_times), dtype=bool)
        
        for i, (t0, t1) in enumerate(batch_times):
            s = int(t0 * SR)
            e = min(int(t1 * SR), n_samples)
            dur = e - s
            if dur > 0 and dur <= MAX_SEG:
                batch_padded[i, :dur] = y_full[s:e]
                valid_mask[i] = True
            elif dur > MAX_SEG:
                # Truncate long segments
                batch_padded[i, :] = y_full[s:s+MAX_SEG]
                valid_mask[i] = True

        # WavLM forward pass
        batch_t = torch.tensor(batch_padded, dtype=torch.float32).to(device)
        with torch.no_grad():
            out = wavlm(batch_t).last_hidden_state  # (batch, seq, 768)
            emb = out.mean(dim=1).squeeze(1)         # (batch, 768)
        
        # Zero out invalid entries
        emb = emb.cpu().numpy()
        emb[~valid_mask] = 0
        all_emb.append(emb)

    return np.vstack(all_emb)

print('Extractor ready (OOM-safe, max_batch=32, MAX_SEG=3s)')


In [ ]:
# Process all videos
t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = OUT_DIR + '/' + vid + '_word_features.npy'
    if os.path.exists(out_file):
        continue

    df = pd.read_csv(label_map[vid])
    word_times, word_labels = [], []
    for _, row in df.iterrows():
        try:
            t0_w, t1_w = parse_timestamp(row['timestamp'])
            word_times.append((t0_w, t1_w))
            word_labels.append(str(row['label']).strip())
        except:
            continue

    feats = extract_video_words(audio_map[vid], word_times)

    if feats is not None and len(feats) > 0:
        np.save(out_file, feats)
        done.add(vid)

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape if feats is not None else "FAIL"} | done={len(done)} | {rate:.0f}/hr')

    if len(done) % 10 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)
print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
# Summary
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(OUT_DIR + '/' + f)
    print(f'  {f}: {d.shape}')
